# Transformers from the Ground Up

## Building Intuition One Step at a Time (TensorFlow / Keras)

This notebook builds the **complete mental model for the Transformer architecture** from first principles — starting with a tiny 3-dimensional semantic space that you can visualise, rotate, and reason about concretely.

Every concept is demonstrated on the same running example:

> **"the cat sat on the mat"**

| Step | Concept | Key Idea |
| ---- | ------- | -------- |
| 1  | Vocabulary + 3D Embeddings     | Words as points in semantic space |
| 2  | The Ordering Problem           | Why bags of words lose meaning |
| 3  | Sinusoidal PE                  | Adding position with sine waves |
| 4  | RoPE                           | Rotating Q/K vectors for relative position |
| 5  | Q, K, V + Attention            | Soft dictionary lookup |
| 6  | Multi-Head Attention           | Parallel attention heads |
| 7  | Feed-Forward + Layer Norm      | Per-token transformation + stabilisation |
| 8  | Full Transformer Block         | All components assembled |
| 9  | Mini Language Model            | End-to-end training from scratch |
| 10 | W_V as Relevance Filter        | What each token contributes to the blend |
| 11 | Causal Triangle                | Layer stacking and last-position richness |
| 12 | Encoder Architecture           | Bidirectional attention — mask=None |
| 13 | Cross-Attention                | Q from decoder, K/V from encoder |
| 14 | DistilGPT-2 Internals          | A real model, cracked open with HuggingFace TF |

**TF/Keras edition:** `nn.Module` → `tf.keras.layers.Layer`, `torch.matmul` → `tf.matmul`, `F.softmax` → `tf.nn.softmax`.

---

## Prerequisite Bridge — From `01-rnns` DL Foundations

| Foundation (from `01-rnns`) | Role in this notebook |
|---|---|
| `tf.keras.layers.Layer` subclassing, `tf.GradientTape` | Every mechanism (`MultiHeadAttention`, `FeedForward`, `TransformerBlock`) is a `Layer`; training uses `GradientTape` |
| Computation graph and `tape.gradient()` | Gradients flow through Q·Kᵀ/√dₖ→softmax→V |
| Tensor shapes: `(batch, features)` → `(batch, seq, dim)` | Every `(B, S, D)` shape annotation assumes comfort with the `01-rnns` shape exercises |

> **If you haven't run `01-rnns`** the GradientTape and shape-reasoning patterns will feel unfamiliar.

In [ ]:
#  Install dependencies (run once)
import subprocess, sys
required = [
    ("numpy","numpy"), ("matplotlib","matplotlib"),
    ("tensorflow","tensorflow"), ("seaborn","seaborn"),
    ("plotly","plotly"), ("transformers","transformers"),
]
for imp, pkg in required:
    try:
        __import__(imp)
        print(f"  ok  {pkg}")
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"  done {pkg}")

In [ ]:
#  Imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math, warnings
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

warnings.filterwarnings('ignore')
tf.random.set_seed(42)
np.random.seed(42)

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})
print(f"TensorFlow {tf.__version__}")
print(f"NumPy      {np.__version__}")

---

## Part 1 — Our Mini Universe: Vocabulary & Embeddings

Before anything else, we need a way to represent words as numbers. This is the job of an **embedding**.

In a real model (e.g. DistilGPT-2), each word lives in a 768-dimensional space. Instead we build a **3-dimensional** semantic space where each axis captures a meaningful property:

| Axis | Meaning | Low (0) | High (1) |
| ---- | ------- | ------- | -------- |
| **dim 0** | Concreteness | abstract (articles) | physical objects (cat, mat) |
| **dim 1** | Animacy | inanimate (mat, fence) | living beings (cat, dog) |
| **dim 2** | Dynamism | static (mat, fence) | action words (ran, jumped) |

In [ ]:
#  Vocabulary + 3D Embeddings
VOCAB = {
    "<PAD>": 0, "<BOS>": 1, "<EOS>": 2,
    "the": 3, "a": 4, "cat": 5, "dog": 6,
    "mat": 7, "fence": 8, "sat": 9, "ran": 10,
    "jumped": 11, "on": 12, "over": 13, "big": 14,
}
IDX2WORD = {v: k for k, v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

E = {
    "<PAD>": [0.00, 0.00, 0.00], "<BOS>": [0.08, 0.08, 0.15], "<EOS>": [0.08, 0.08, 0.15],
    "the":   [0.05, 0.04, 0.08], "a":     [0.05, 0.04, 0.08],
    "cat":   [0.91, 0.94, 0.38], "dog":   [0.88, 0.92, 0.55],
    "mat":   [0.96, 0.04, 0.04], "fence": [0.93, 0.03, 0.03],
    "sat":   [0.34, 0.18, 0.78], "ran":   [0.28, 0.12, 0.96],
    "jumped":[0.30, 0.14, 0.98], "on":    [0.14, 0.04, 0.18],
    "over":  [0.17, 0.04, 0.24], "big":   [0.44, 0.04, 0.09],
}

# Build embedding matrix as tf.constant (lookup table)
embedding_matrix = tf.constant(
    [E[IDX2WORD[i]] for i in range(VOCAB_SIZE)], dtype=tf.float32
)  # shape: (VOCAB_SIZE, 3)

SENTENCE = "the cat sat on the mat"
TOKENS = SENTENCE.split()
TOKEN_IDS = [VOCAB[w] for w in TOKENS]
SEQ_LEN = len(TOKENS)

print(f"Vocab size       : {VOCAB_SIZE}")
print(f"Embedding shape  : {embedding_matrix.shape}  (vocab x 3D)")
print(f"Running sentence : {SENTENCE!r}")
print(f"Token IDs        : {TOKEN_IDS}")
print()
print("3D semantic axes -> Concreteness, Animacy, Dynamism:")
for word, vec in list(E.items())[3:]:
    print(f"  {word:<10} [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")

In [ ]:
#  Keras Embedding layer: manual lookup vs layers.Embedding
# For the toy 3D space, we use tf.nn.embedding_lookup (same as manual indexing).
# In a real model, layers.Embedding is trained end-to-end.

emb_layer_demo = layers.Embedding(VOCAB_SIZE, 3, trainable=False)
emb_layer_demo.build((None,))
emb_layer_demo.set_weights([embedding_matrix.numpy()])

ids = tf.constant([TOKEN_IDS])  # (1, 6)
looked_up = emb_layer_demo(ids)   # (1, 6, 3)
print(f"Embedding lookup shape: {looked_up.shape}  -> (batch=1, seq=6, dim=3)")
print()
print("Token embeddings for sentence (Concreteness, Animacy, Dynamism):")
for i, tok in enumerate(TOKENS):
    v = looked_up[0, i].numpy()
    print(f"  [{i}] {tok:<8}  [{v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f}]")

In [ ]:
#  Interactive 3D Vocabulary Visualisation
CATEGORIES = {
    "Article":        (["the", "a"],            "#636EFA"),
    "Animate Noun":   (["cat", "dog"],           "#00CC96"),
    "Inanimate Noun": (["mat", "fence"],         "#AB63FA"),
    "Verb":           (["sat", "ran", "jumped"], "#EF553B"),
    "Preposition":    (["on", "over"],           "#FFA15A"),
    "Adjective":      (["big"],                  "#19D3F3"),
}

if HAS_PLOTLY:
    fig = go.Figure()
    for cat, (words, color) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='markers+text', text=words,
            textposition='top center', name=cat,
            marker=dict(size=10, color=color, opacity=0.85)))
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    fig.add_trace(go.Scatter3d(x=sx, y=sy, z=sz, mode='lines', name='Sentence path',
        line=dict(color='gold', width=4, dash='dot')))
    fig.update_layout(
        title=dict(text='<b>3D Semantic Embedding Space</b> — drag to rotate', x=0.5),
        scene=dict(xaxis_title='Concreteness', yaxis_title='Animacy', zaxis_title='Dynamism'),
        width=820, height=560)
    fig.show()
else:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection='3d')
    cmap = {'Article': 'royalblue', 'Animate Noun': 'mediumseagreen',
            'Inanimate Noun': 'mediumpurple', 'Verb': 'tomato',
            'Preposition': 'darkorange', 'Adjective': 'deepskyblue'}
    for cat, (words, _) in CATEGORIES.items():
        xs, ys, zs = zip(*[E[w] for w in words])
        ax.scatter(xs, ys, zs, s=90, label=cat, color=cmap[cat], alpha=0.9)
        for w in words:
            ax.text(E[w][0], E[w][1], E[w][2], f' {w}', fontsize=9)
    sx, sy, sz = zip(*[E[w] for w in TOKENS])
    ax.plot(sx, sy, sz, 'o--', color='gold', lw=2, label='Sentence path')
    ax.set_xlabel('Concreteness'); ax.set_ylabel('Animacy'); ax.set_zlabel('Dynamism')
    ax.set_title('3D Semantic Embedding Space'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print('Tip: pip install plotly for an interactive, rotatable version')

### Tokenisation

A **tokeniser** converts a raw string into integer IDs the model can process. In our toy system, one word = one token. Production models use sub-word tokenisation (BPE / SentencePiece).

In [ ]:
#  Tokeniser
def encode(text: str, add_bos=False, add_eos=False):
    ids = [VOCAB.get(w, VOCAB["<PAD>"]) for w in text.lower().split()]
    if add_bos: ids = [VOCAB["<BOS>"]] + ids
    if add_eos: ids = ids + [VOCAB["<EOS>"]]
    return ids

def decode(ids):
    return " ".join(IDX2WORD.get(i, "<?>") for i in ids)

phrase = "the big cat jumped over the fence"
enc = encode(phrase)
print(f"Input  : {phrase!r}")
print(f"Encoded: {enc}")
print(f"Decoded: {decode(enc)!r}")
print()
enc2 = encode(phrase, add_bos=True, add_eos=True)
print(f"With BOS/EOS: {enc2}")

---

## Attention: First Contact

Before positions, before Q/K/V projections, before multi-head — the beating heart of the transformer is one simple idea:

> **Every token looks at every other token and builds a weighted average of them, where the weights answer "how much do I care about you?"**

**Running it on our sentence:** `"the cat sat on the mat"`. Query = `"cat"`. No positions, no learned projections — just raw dot products of the 3D semantic vectors.

In [ ]:
#  Minimal attention: raw dot products of embeddings
emb_np = embedding_matrix.numpy()[TOKEN_IDS]   # (S, 3)
S_min = len(TOKENS)
QUERY = "cat"
qi = TOKENS.index(QUERY)

# Step-by-step: scores -> softmax weights -> weighted sum
scores_min = emb_np @ emb_np[qi]          # raw dot products (S,)
w_min = np.exp(scores_min - scores_min.max())
w_min /= w_min.sum()                        # softmax normalise
output_min = (w_min[:, None] * emb_np).sum(0)  # (3,) context vector

print(f'Minimal attention: query = key = value = raw embedding (no W_Q/W_K/W_V)')
print(f'Query token: "{QUERY}" at position {qi}')
print()
print(f'{"Token":<8} {"Score":>7} {"Weight":>7}')
print('  ' + '-' * 24)
for j, (tok, s, w) in enumerate(zip(TOKENS, scores_min, w_min)):
    bar = '*' * int(w * 30)
    print(f'  {tok:<8} {s:>7.3f} {w:>7.3f}  {bar}')
print()
top3 = w_min.argsort()[::-1][:3]
print('"cat" attends most to: ' + ', '.join(f'{TOKENS[j]} ({w_min[j]:.0%})' for j in top3))
print(f'Output context vector: [{output_min[0]:.3f}, {output_min[1]:.3f}, {output_min[2]:.3f}]')

#### What just happened — and what's missing

`"cat"` pulled most strongly toward **itself** and **`"dog"`** — its semantic neighbours. Attention found *meaning* without anyone hand-coding grammar.

But look closely at what we **never used**: *position*. Query, key and value were the raw embeddings. Shuffle the sentence and `"cat"` keeps the exact same neighbours. **Attention, on its own, is position-blind.**

---

## Part 2 — The Ordering Problem

What happens if we just **sum or average** the token vectors?

> "the cat sat on the mat"  
> "mat the on sat the cat" — shuffled nonsense

Both sentences contain exactly the same words. Their mean-pooled embedding is **identical** — the model cannot tell them apart. Position information is load-bearing.

In [ ]:
#  Bag of Words — loses all positional information (TF version)
def bag_of_words(sentence: str):
    ids = tf.constant([encode(sentence)], dtype=tf.int32)
    vecs = tf.nn.embedding_lookup(embedding_matrix, ids)  # (1, S, 3)
    return tf.reduce_mean(vecs, axis=1)[0].numpy()         # (3,)

sentences = [
    "the cat sat on the mat",
    "mat the on sat the cat",
    "sat cat mat on the the",
]
print('Mean-pooled vectors (all contain the same words):')
for s in sentences:
    v = bag_of_words(s)
    print(f"  {s!r:<42}  [{v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}]")

all_same = all(
    np.allclose(bag_of_words(sentences[0]), bag_of_words(s))
    for s in sentences[1:]
)
print(f"\nAll three vectors identical: {all_same}")
print()
print("  -> A model with no positional encoding treats meaningful sentences")
print("     and complete nonsense as the SAME input.  We need PE.")

---

## Part 3 — Positional Encoding

### 3a. Sinusoidal PE (original Transformer, "Attention Is All You Need")

$$PE_{(m,\, 2i)} = \sin\!\left(\frac{m}{10000^{2i/d}}\right) \qquad PE_{(m,\, 2i+1)} = \cos\!\left(\frac{m}{10000^{2i/d}}\right)$$

Each dimension pair oscillates at a different frequency — a unique fingerprint for every position.

In TensorFlow we compute this with pure `tf.Tensor` arithmetic — no lookup table, no trainable parameters.

In [ ]:
#  Sinusoidal Positional Encoding (pure tf.Tensor arithmetic)
def sinusoidal_pe(seq_len: int, d_model: int) -> tf.Tensor:
    """Classic additive positional encoding (Vaswani et al. 2017).
    Returns tf.Tensor of shape (seq_len, d_model).
    Works for both even and odd d_model.
    """
    positions = tf.cast(tf.range(seq_len)[:, tf.newaxis], tf.float32)  # (S, 1)
    half_d = (d_model + 1) // 2                                         # ceil(d/2)
    dims = tf.cast(tf.range(half_d), tf.float32)                        # (half_d,)
    freqs = 1.0 / tf.pow(10000.0, 2.0 * dims / tf.cast(d_model, tf.float32))  # (half_d,)
    angles = positions * freqs                                           # (S, half_d)
    sin_part = tf.math.sin(angles)  # (S, half_d)
    cos_part = tf.math.cos(angles)  # (S, half_d)
    # Interleave sin/cos: even dims -> sin, odd dims -> cos
    pe = tf.reshape(
        tf.stack([sin_part, cos_part], axis=2),  # (S, half_d, 2)
        [seq_len, 2 * half_d]                     # (S, 2*half_d)
    )
    return pe[:, :d_model]  # trim last col if d_model is odd

D_VIS = 16
pe_matrix = sinusoidal_pe(SEQ_LEN, D_VIS).numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
im = ax.imshow(pe_matrix, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(D_VIS))
ax.set_xticklabels([f'd{i}' for i in range(D_VIS)], fontsize=8, rotation=45)
ax.set_yticks(range(SEQ_LEN)); ax.set_yticklabels(TOKENS, fontsize=10)
ax.set_title('Sinusoidal PE — our sentence')
ax.set_xlabel('Embedding dimension'); ax.set_ylabel('Token position')
plt.colorbar(im, ax=ax)

ax2 = axes[1]
pe_long = sinusoidal_pe(50, D_VIS).numpy()
for i in [0, 2, 6, 14]:
    ax2.plot(pe_long[:, i], label=f'dim {i} — {"fast" if i < 4 else "slow"}', lw=1.8)
ax2.set_title('PE signal per dimension over 50 positions')
ax2.set_xlabel('Token position'); ax2.set_ylabel('PE value')
ax2.legend(fontsize=8); ax2.set_ylim(-1.1, 1.1)

plt.suptitle('Sinusoidal Positional Encoding', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

pe_3d = sinusoidal_pe(SEQ_LEN, 3).numpy()
emb_vecs = embedding_matrix.numpy()[TOKEN_IDS]
enriched = emb_vecs + pe_3d
print('After adding 3D sinusoidal PE:')
for i, w in enumerate(TOKENS):
    print(f'[{i}] {w:<8}  emb=[{emb_vecs[i,0]:+.3f},{emb_vecs[i,1]:+.3f},{emb_vecs[i,2]:+.3f}]  '
          f'pe=[{pe_3d[i,0]:+.3f},{pe_3d[i,1]:+.3f},{pe_3d[i,2]:+.3f}]  '
          f'sum=[{enriched[i,0]:+.3f},{enriched[i,1]:+.3f},{enriched[i,2]:+.3f}]')


### 3b. RoPE — Rotary Positional Embeddings

RoPE **rotates** the Query and Key vectors just before the dot-product, by an angle that depends on absolute position. The rotation cancels in a relative way — only the gap $m - n$ survives in the dot product.

$$\theta_i = \frac{1}{10000^{2i/d}} \qquad \text{RoPE}(x, m)_{2i:2i+2} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

In [ ]:
#  RoPE theta values — frequency decay visualisation
D_ROPE = 6
half = D_ROPE // 2
thetas = np.array([1.0 / (10000 ** (2 * i / D_ROPE)) for i in range(half)])
print(f'theta values for d={D_ROPE}: {thetas}')

steps = 50
angles = np.outer(np.arange(steps), thetas)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for i in range(half):
    ax.plot(np.cos(angles[:, i]), label=f'Pair {i} (theta={thetas[i]:.4f})', lw=1.8)
ax.set_xlabel('Token position'); ax.set_ylabel('cos(m * theta_i)')
ax.set_title('RoPE: cosine component per dimension pair over 50 positions')
ax.legend(fontsize=8); ax.set_ylim(-1.1, 1.1)

ax2 = axes[1]
for i in range(half):
    ax2.plot(range(steps), angles[:, i] % (2 * np.pi), label=f'Pair {i}', lw=1.8)
ax2.set_xlabel('Token position'); ax2.set_ylabel('angle mod 2pi')
ax2.set_title('Accumulated rotation angle\nPair 0 spins fastest')
ax2.legend(fontsize=8)
plt.suptitle('RoPE theta values: high-frequency pairs capture local position', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
#  RoPE implementation + relative-distance proof (pure numpy)
def rope_rotate(x: np.ndarray, m: int, thetas_arr: np.ndarray) -> np.ndarray:
    """Rotate vector x (shape: d) at token position m using RoPE."""
    x_rot = np.array(x, dtype=np.float32).copy()
    for i, th in enumerate(thetas_arr):
        angle = m * th
        c, s = math.cos(angle), math.sin(angle)
        a, b = x_rot[2*i], x_rot[2*i+1]
        x_rot[2*i]   = a*c - b*s
        x_rot[2*i+1] = a*s + b*c
    return x_rot

thetas_demo = np.array([1.0 / (10000 ** (2*i/D_ROPE)) for i in range(D_ROPE//2)])
# Tile 3-D semantic embeddings → 6-D so that D_ROPE=6 (3 rotation pairs) fits
cat_emb = np.tile(embedding_matrix.numpy()[VOCAB['cat']], 2)[:D_ROPE]
mat_emb = np.tile(embedding_matrix.numpy()[VOCAB['mat']], 2)[:D_ROPE]

print('Relative-distance property of RoPE:')
print('  cat at pos m, mat at pos n: Q_cat * K_mat depends only on (m-n)')
print()
for gap in [1, 2, 3]:
    results = []
    for base in [0, 1, 2, 3]:
        q = rope_rotate(cat_emb, base, thetas_demo)
        k = rope_rotate(mat_emb, base + gap, thetas_demo)
        results.append(float(q @ k))
    print(f'  gap={gap}: dot products = {[f"{r:.4f}" for r in results]}  '
          f'(all equal -> {np.allclose(results, results[0], atol=1e-5)})')

print()
print('  -> RoPE guarantees: the dot product depends ONLY on (m-n).')


---

## Part 4 — Queries, Keys & Values

The transformer's attention mechanism is a **soft dictionary lookup**.

| Component | Intuition | Created by |
| --------- | --------- | ---------- |
| **Q** (Query) | "What am I looking for?" | `Dense(d_k)(x)` |
| **K** (Key)   | "What do I advertise?"   | `Dense(d_k)(x)` |
| **V** (Value) | "What do I contribute?"  | `Dense(d_k)(x)` |

$W_Q$, $W_K$, $W_V$ are **learned** projection matrices (implemented as `layers.Dense` without bias).

In [ ]:
#  Scaled Dot-Product Attention — built with tf.matmul + tf.nn.softmax
D_MODEL = 3   # toy dimension (same as embedding space)

np.random.seed(7)
W_Q = tf.constant(np.random.randn(D_MODEL, D_MODEL).astype(np.float32) * 0.5)
W_K = tf.constant(np.random.randn(D_MODEL, D_MODEL).astype(np.float32) * 0.5)
W_V = tf.constant(np.random.randn(D_MODEL, D_MODEL).astype(np.float32) * 0.5)

embs = tf.constant(embedding_matrix.numpy()[TOKEN_IDS])   # (6, 3)
Q = embs @ W_Q   # (6, 3)
K = embs @ W_K
V = embs @ W_V

def scaled_dot_product_attention(Q_in, K_in, V_in, mask=None):
    """Scaled dot-product attention.  Returns (output, attn_weights).
    Q_in, K_in, V_in: (seq_len, d_k)
    """
    d_k = tf.cast(tf.shape(Q_in)[-1], tf.float32)
    scores = tf.matmul(Q_in, K_in, transpose_b=True) / tf.math.sqrt(d_k)
    print(f'  Raw score matrix (QK^T / sqrt(d_k)):\n  {scores.numpy().round(3)}')
    if mask is not None:
        scores = scores + (1.0 - tf.cast(mask, tf.float32)) * (-1e9)
    attn_w = tf.nn.softmax(scores, axis=-1)
    out = tf.matmul(attn_w, V_in)
    return out, attn_w

print('=== Step-by-step attention on our sentence ===')
print(f'Q shape: {Q.shape}  K shape: {K.shape}  V shape: {V.shape}\n')

out, attn_w = scaled_dot_product_attention(Q, K, V)
print(f'\n  Attention weight matrix (row = query token, col = key token):')
print(f'  Tokens: {TOKENS}')
print(f'  {attn_w.numpy().round(3)}')
print(f'\n  Output shape: {out.shape}')

In [ ]:
#  Attention heatmap visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
w = attn_w.numpy()
sns.heatmap(w, ax=ax, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
            cbar_kws={'label': 'attention weight'})
ax.set_title('Bidirectional Attention (encoder-style)')
ax.set_xlabel('Key token'); ax.set_ylabel('Query token'); ax.tick_params(axis='x', rotation=30)

ax2 = axes[1]
# Causal mask: lower-triangular (1 = attend, 0 = block)
causal_mask_vis = tf.linalg.band_part(tf.ones((SEQ_LEN, SEQ_LEN)), -1, 0)
_, attn_w_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask_vis)
wc = attn_w_causal.numpy()
sns.heatmap(wc, ax=ax2, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
            cbar_kws={'label': 'attention weight'})
ax2.set_title('Causal Attention (decoder-style)\nupper triangle masked to -inf')
ax2.set_xlabel('Key token'); ax2.set_ylabel('Query token'); ax2.tick_params(axis='x', rotation=30)

plt.suptitle('Bidirectional vs Causal Attention', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

### Your turn — attention

You've watched attention; now drive it. Change `my_query` and **predict the top attention target before you run it**.

In [ ]:
#  EXERCISE 1 — attention by hand
# Change `my_query` to any token in TOKENS and PREDICT its top attention target BEFORE running.
# TOKENS = ['the', 'cat', 'sat', 'on', 'the', 'mat']
my_query = 'cat'   # try 'sat', 'mat', 'on', ...

qi = TOKENS.index(my_query)
scores_ex = tf.matmul(Q, K, transpose_b=True) / tf.math.sqrt(tf.cast(D_MODEL, tf.float32))
w_ex = tf.nn.softmax(scores_ex, axis=-1).numpy()[qi]

print(f'"{my_query}" (position {qi}) attends most to:')
for r in w_ex.argsort()[::-1][:3]:
    print(f'   {TOKENS[r]:<8} (pos {r})  weight={w_ex[r]:.3f}')

---

## Part 5 — Multi-Head Attention

**Multi-Head Attention** (MHA) runs $H$ parallel attention heads:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \cdot W_O$$

**Why multiple heads?** Each head can specialise on a different relationship type.

### Hand-Built Version (tf.Tensor ops)

In [ ]:
#  Hand-built MultiHeadAttention as a tf.keras.layers.Layer
D_WORK = 16    # functional model dimension
NUM_HEADS = 2  # attention heads
D_HEAD = D_WORK // NUM_HEADS   # 8 per head
D_FF = 32      # feed-forward hidden size

class MultiHeadAttentionManual(tf.keras.layers.Layer):
    """Hand-built MHA using tf.matmul + tf.nn.softmax (mirrors the equations directly)."""

    def __init__(self, d_model, n_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        # Projections: bias=False to match standard Transformer convention
        self.W_Q = layers.Dense(d_model, use_bias=False)
        self.W_K = layers.Dense(d_model, use_bias=False)
        self.W_V = layers.Dense(d_model, use_bias=False)
        self.W_O = layers.Dense(d_model, use_bias=False)

    def call(self, x, mask=None, training=False):
        B = tf.shape(x)[0]
        S = tf.shape(x)[1]
        # Project and split heads: (B, S, d_model) -> (B, n_heads, S, d_head)
        def split_heads(t):
            t = tf.reshape(t, [B, S, self.n_heads, self.d_head])
            return tf.transpose(t, perm=[0, 2, 1, 3])  # (B, H, S, d_head)
        Q_mh = split_heads(self.W_Q(x))
        K_mh = split_heads(self.W_K(x))
        V_mh = split_heads(self.W_V(x))
        # Scaled dot-product: (B, H, S, S)
        d_k = tf.cast(self.d_head, tf.float32)
        scores = tf.matmul(Q_mh, K_mh, transpose_b=True) / tf.math.sqrt(d_k)
        if mask is not None:
            # mask: (S, S) lower-triangular float (1=attend, 0=block)
            scores = scores + (1.0 - tf.cast(mask, tf.float32)) * (-1e9)
        attn_w = tf.nn.softmax(scores, axis=-1)   # (B, H, S, S)
        out = tf.matmul(attn_w, V_mh)             # (B, H, S, d_head)
        # Merge heads: (B, H, S, d_head) -> (B, S, d_model)
        out = tf.transpose(out, perm=[0, 2, 1, 3])
        out = tf.reshape(out, [B, S, self.d_model])
        return self.W_O(out), attn_w

# Demo
tf.random.set_seed(42)
mha_manual = MultiHeadAttentionManual(D_WORK, NUM_HEADS)

proj_demo = layers.Dense(D_WORK, use_bias=False)
x_work = proj_demo(embs[tf.newaxis, :, :])  # (1, 6, 16)

mha_out, head_weights = mha_manual(x_work)
print(f'MHA output shape     : {mha_out.shape}')
print(f'Head weights shape   : {head_weights.shape}   (batch, n_heads, seq, seq)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h in range(NUM_HEADS):
    ax = axes[h]
    w_h = head_weights[0, h].numpy()
    sns.heatmap(w_h, ax=ax, annot=True, fmt='.2f', cmap='Purples',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(f'Head {h} attention weights')
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Multi-Head Attention — each head learns a different relationship', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

### Using `tf.keras.layers.MultiHeadAttention` (the built-in)

TensorFlow provides `layers.MultiHeadAttention(num_heads=H, key_dim=d_k)` which handles all the projection and splitting internally. The API is:

```python
mha = layers.MultiHeadAttention(num_heads=H, key_dim=d_k)
output, weights = mha(query, value, key, return_attention_scores=True)
# For self-attention: query = value = key = x
# For causal masking: use_causal_mask=True  (TF 2.12+)
```

The built-in layer uses a different weight layout internally but is mathematically identical to our hand-built version.

In [ ]:
#  tf.keras.layers.MultiHeadAttention — built-in comparison
mha_keras = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=D_HEAD,
                                       dropout=0.0)

# Self-attention (query = value = key = x)
keras_out, keras_attn = mha_keras(
    query=x_work, value=x_work, key=x_work,
    return_attention_scores=True
)
print(f'Keras MHA output shape      : {keras_out.shape}')
print(f'Keras MHA attention shape   : {keras_attn.shape}  (batch, heads, seq, seq)')

# Causal self-attention using use_causal_mask
mha_causal = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=D_HEAD)
causal_out, causal_attn = mha_causal(
    query=x_work, value=x_work, key=x_work,
    use_causal_mask=True, return_attention_scores=True
)
print(f'Causal MHA output shape     : {causal_out.shape}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for h, (ax, w, title) in enumerate(zip(axes,
    [keras_attn[0, 0].numpy(), causal_attn[0, 0].numpy()],
    ['Keras MHA — bidirectional (Head 0)', 'Keras MHA — causal (Head 0)'])):
    sns.heatmap(w, ax=ax, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=10); ax.tick_params(axis='x', rotation=30)
plt.suptitle('tf.keras.layers.MultiHeadAttention — bidirectional vs causal', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
#  Proving the multi-head claim — 2 heads capture 2 different relations
S = SEQ_LEN
V_shared = embs.numpy() @ np.random.randn(3, 3).astype(np.float32)

# Relation P (positional): attend to PREVIOUS token
prev_idx = np.array([max(i-1, 0) for i in range(S)])
scores_pos = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S): scores_pos[i, prev_idx[i]] = 9.0
A_pos = tf.nn.softmax(tf.constant(scores_pos), axis=-1).numpy()

# Relation C (content): attend to most SEMANTICALLY SIMILAR token
sim = embs.numpy() @ embs.numpy().T
np.fill_diagonal(sim, -1e9)
near_idx = sim.argmax(-1)
scores_con = np.full((S, S), -9.0, dtype=np.float32)
for i in range(S): scores_con[i, near_idx[i]] = 9.0
A_con = tf.nn.softmax(tf.constant(scores_con), axis=-1).numpy()

corr = np.corrcoef(A_pos.flatten(), A_con.flatten())[0, 1]
print(f'Correlation between the two head patterns: {corr:+.3f}   (~0 -> different information)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, A, title, cmap in [
    (axes[0], A_pos, 'Head P — positional (attend to previous token)', 'Greens'),
    (axes[1], A_con, 'Head C — content (attend to most similar token)', 'Purples'),
]:
    sns.heatmap(A, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5, cbar=False)
    ax.set_title(title, fontsize=10); ax.set_xlabel('Key'); ax.set_ylabel('Query')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Two heads, two DIFFERENT relations', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
#  One head cannot do both — measuring reconstruction error
target_prev = V_shared[prev_idx]
target_near = V_shared[near_idx]
out_pos = A_pos @ V_shared
out_con = A_con @ V_shared

def mse(a, b): return float(((a - b) ** 2).mean())

print('Reconstruction error (lower = that relation is captured):')
print(f'  Head P -> previous-token target : {mse(out_pos, target_prev):.4f}   <- nails it')
print(f'  Head P -> similar-token  target : {mse(out_pos, target_near):.4f}   <- misses it')
print(f'  Head C -> previous-token target : {mse(out_con, target_prev):.4f}   <- misses it')
print(f'  Head C -> similar-token  target : {mse(out_con, target_near):.4f}   <- nails it')
print()
print('  -> A single head serves ONE relation well, never both.')
print('  -> Concatenating [Head P ; Head C] delivers BOTH targets in parallel.')

---

## Part 6 — Feed-Forward Network & Layer Normalisation

### Feed-Forward Network (FFN)

$$\text{FFN}(x) = \text{GELU}(x W_1 + b_1)\, W_2 + b_2$$

Expands by 4× then projects back. Adds non-linear transformation capacity.

### Layer Normalisation

Applied **before** each sub-layer (Pre-LN style). `layers.LayerNormalization(epsilon=1e-6)` normalises each token's vector to zero mean and unit variance.

In [ ]:
#  FeedForward + LayerNorm visualisation (Keras version)
class FeedForward(tf.keras.layers.Layer):
    """Position-wise FFN: Dense(d_model->d_ff, gelu) -> Dense(d_ff->d_model)."""
    def __init__(self, d_model, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = layers.Dense(d_ff, activation='gelu')
        self.dense2 = layers.Dense(d_model)
    def call(self, x, training=False):
        return self.dense2(self.dense1(x))

tf.random.set_seed(42)
ffn = FeedForward(D_WORK, D_FF)
norm = layers.LayerNormalization(epsilon=1e-6)

x_raw = x_work[0]             # (6, 16)
x_after = ffn(x_raw)          # (6, 16) raw FFN output
x_normed = norm(x_after)      # (6, 16) after LayerNorm

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, data, title in zip(axes, [x_raw, x_after, x_normed],
                           ['Input to FFN', 'FFN output (raw)', 'After LayerNorm']):
    data_np = data.numpy()
    for j, token in enumerate(TOKENS):
        vals = data_np[j]
        ax.plot(vals, alpha=0.7, label=f'{token}  mu={vals.mean():.2f}')
    ax.set_title(title); ax.set_xlabel('Hidden dimension'); ax.set_ylabel('Activation value')
    ax.legend(fontsize=7); ax.axhline(0, color='black', lw=0.5, ls='--')
plt.suptitle('FFN activations before and after LayerNorm', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('LayerNorm centres and normalises each token slice.')
x_normed_np = x_normed.numpy()
for j, token in enumerate(TOKENS):
    v = x_normed_np[j]
    print(f'  {token:<8}  mean={v.mean():+.4f}  std={v.std():.4f}')

---

## Part 7 — Full Transformer Block & Residual Connections

A single **Transformer Block** wires MHA + FFN together with layer norm and residuals:

```
x --> LayerNorm --> MHA --> (+x) --> LayerNorm --> FFN --> (+x) --> output
```

Stack $L$ of these blocks = the full encoder/decoder stack.

In [ ]:
#  TransformerBlock as a tf.keras.layers.Layer
class TransformerBlock(tf.keras.layers.Layer):
    """Pre-LN Transformer block: LN->MHA->residual, LN->FFN->residual."""

    def __init__(self, d_model, n_heads, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.mha   = layers.MultiHeadAttention(num_heads=n_heads,
                                               key_dim=d_model // n_heads,
                                               dropout=0.0)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn   = FeedForward(d_model, d_ff)

    def call(self, x, mask=None, training=False):
        # mask: None (bidirectional) or lower-triangular bool (causal)
        normed = self.norm1(x)
        attn_out = self.mha(normed, normed, attention_mask=mask, training=training)
        x = x + attn_out
        x = x + self.ffn(self.norm2(x), training=training)
        return x

tf.random.set_seed(42)
block1 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)
block2 = TransformerBlock(D_WORK, NUM_HEADS, D_FF)

x0 = x_work   # (1, 6, 16)
x1 = block1(x0)
x2 = block2(x1)

print('Input -> Block 1 -> Block 2:')
for label, xi in [('x0', x0), ('x1', x1), ('x2', x2)]:
    print(f'  {label}: {xi.shape}  norm={tf.norm(xi).numpy():.3f}')

fig, ax = plt.subplots(figsize=(8, 4))
norms = {
    'Layer 0 (input)': tf.norm(x0[0], axis=-1).numpy(),
    'Layer 1 output':  tf.norm(x1[0], axis=-1).numpy(),
    'Layer 2 output':  tf.norm(x2[0], axis=-1).numpy(),
}
x_pos = np.arange(SEQ_LEN); width = 0.25
for k, (label, vals) in enumerate(norms.items()):
    ax.bar(x_pos + k*width, vals, width, label=label, alpha=0.85)
ax.set_xticks(x_pos + width); ax.set_xticklabels(TOKENS)
ax.set_ylabel('Representation L2 norm')
ax.set_title('Token representations grow through transformer blocks')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
#  Why residual connections make depth trainable
# Simulate 24 layers with and without skip connections using tf.GradientTape

DEPTH = 24; d = D_WORK

class ProbeLayer(tf.keras.layers.Layer):
    def __init__(self, d_in, residual, **kwargs):
        super().__init__(**kwargs)
        self.lin = layers.Dense(d_in)
        self.residual = residual
    def call(self, x):
        y = tf.math.tanh(self.lin(x))
        return x + y if self.residual else y

def gradient_norms(residual):
    tf.random.set_seed(0)
    probe_layers = [ProbeLayer(d, residual) for _ in range(DEPTH)]
    x_p = tf.random.normal([1, d])
    with tf.GradientTape() as tape:
        h = x_p
        for layer in probe_layers: h = layer(h)
        loss_p = tf.reduce_mean(h ** 2)
    grads = tape.gradient(loss_p, [l.lin.kernel for l in probe_layers])
    return [float(tf.norm(g)) for g in grads]

g_res   = gradient_norms(True)
g_plain = gradient_norms(False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, DEPTH+1), g_plain, 'o-', color='tomato',   lw=2, label='WITHOUT residual')
ax.plot(range(1, DEPTH+1), g_res,   'o-', color='seagreen', lw=2, label='WITH residual')
ax.set_yscale('log')
ax.set_xlabel('Layer (1 = furthest from loss)')
ax.set_ylabel('|gradient| reaching this layer  (log scale)')
ax.set_title('Residual connections keep gradients alive all the way to layer 1')
ax.legend(); plt.tight_layout(); plt.show()

print(f'  WITHOUT residual, layer 1: {g_plain[0]:.2e}   <- vanished')
print(f'  WITH    residual, layer 1: {g_res[0]:.2e}   <- healthy')

---

## Part 8 — Mini Language Model: Training & Inference

We now wire everything together into a **Mini Language Model** — a decoder-only Transformer that learns to predict the next token. This is the architecture of GPT, LLaMA, Mistral etc.

**Training task**: given a context window, predict the next token.

In [ ]:
#  MiniLM — decoder-only Transformer LM (Keras subclass)
class MiniLM(tf.keras.Model):
    """
    Decoder-only transformer LM.
    Architecture: Embedding + sinusoidal PE -> N x TransformerBlock (causal) -> lm_head
    """
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq=64, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.pe = sinusoidal_pe(max_seq, d_model)   # (max_seq, d_model), fixed
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.norm_out = layers.LayerNormalization(epsilon=1e-6)
        self.lm_head = layers.Dense(vocab_size, use_bias=False)

    def call(self, token_ids, training=False):
        """token_ids: (B, S) int tensor -> logits: (B, S, vocab_size)"""
        S = tf.shape(token_ids)[1]
        x = self.token_emb(token_ids) + self.pe[:S]   # (B, S, d_model)
        # Causal mask: (S, S) bool where True = allowed to attend
        i = tf.range(S)[:, tf.newaxis]
        j = tf.range(S)[tf.newaxis, :]
        causal_mask = tf.cast(i >= j, tf.bool)  # lower triangular
        for block in self.blocks:
            x = block(x, mask=causal_mask, training=training)
        x = self.norm_out(x)
        return self.lm_head(x)   # weight tying not used here for clarity

tf.random.set_seed(42)
model_demo = MiniLM(vocab_size=VOCAB_SIZE, d_model=D_WORK, n_heads=NUM_HEADS,
                    d_ff=D_FF, n_layers=2)
_ = model_demo(tf.constant([TOKEN_IDS]))
n_params = sum(np.prod(v.shape) for v in model_demo.trainable_variables)
print(f'MiniLM -> {n_params:,} trainable parameters')
for v in model_demo.trainable_variables:
    print(f'  {v.name:<50} {tuple(v.shape)}')

In [ ]:
#  Training data — (context, next_token) pairs
full_corpus = [
    "the cat sat on the mat",
    "the dog ran over the fence",
    "a big cat jumped over the fence",
    "a dog sat on the mat",
    "the cat jumped over the fence",
    "the big dog ran on the mat",
]
TRAIN_PAIRS = []
for sentence in full_corpus:
    ids = encode(sentence)
    for end in range(1, len(ids)):
        TRAIN_PAIRS.append((ids[:end], ids[end]))
print(f'Training pairs: {len(TRAIN_PAIRS)}')
print('\nFirst 6 examples:')
for ctx, tgt in TRAIN_PAIRS[:6]:
    print(f'  {[IDX2WORD[i] for i in ctx]}  ->  "{IDX2WORD[tgt]}"')

In [ ]:
#  Training loop with tf.GradientTape
def pad_collate(pairs, pad_id=0):
    max_len = max(len(ctx) for ctx, _ in pairs)
    xs, ys = [], []
    for ctx, tgt in pairs:
        pad = [pad_id] * (max_len - len(ctx))
        xs.append(pad + ctx)
        ys.append(tgt)
    return (tf.constant(xs, dtype=tf.int32),
            tf.constant(ys, dtype=tf.int32))

tf.random.set_seed(42)
model = MiniLM(VOCAB_SIZE, D_WORK, NUM_HEADS, D_FF, n_layers=2)
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-3)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

EPOCHS = 300
x_train, y_train = pad_collate(TRAIN_PAIRS)
loss_history, acc_history = [], []

for epoch in range(EPOCHS):
    with tf.GradientTape() as tape:
        logits = model(x_train, training=True)
        last_logits = logits[:, -1, :]   # predict from last position
        loss = loss_fn(y_train, last_logits)
    grads = tape.gradient(loss, model.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 1.0)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if (epoch + 1) % 10 == 0:
        preds = tf.argmax(last_logits, axis=-1, output_type=tf.int32)
        acc = tf.reduce_mean(tf.cast(preds == y_train, tf.float32)).numpy()
        loss_history.append(loss.numpy())
        acc_history.append(acc)
        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:4d} | loss={loss.numpy():.4f} | acc={acc:.2%}')
print('\nTraining complete.')

In [ ]:
#  Training curves
epochs_logged = list(range(10, EPOCHS + 1, 10))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(epochs_logged, loss_history, color='royalblue', lw=2)
ax.fill_between(epochs_logged, loss_history, alpha=0.15, color='royalblue')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss'); ax.set_title('Training Loss')

ax2 = axes[1]
ax2.plot(epochs_logged, [a * 100 for a in acc_history], color='mediumseagreen', lw=2)
ax2.fill_between(epochs_logged, [a * 100 for a in acc_history], alpha=0.15, color='mediumseagreen')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)'); ax2.set_title('Next-Token Prediction Accuracy')
ax2.set_ylim(0, 105); ax2.axhline(100, color='grey', ls='--', lw=0.8)

plt.suptitle('MiniLM Training Progress', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

---

## Part 9 — Inference: Autoregressive Token Generation

At inference time a language model generates text **one token at a time**:

1. Feed the current context into the model  
2. Take the logits at the **last position**  
3. Apply temperature scaling + softmax  
4. Sample (or argmax = greedy)  
5. Append the new token → go to step 1

In [ ]:
#  Autoregressive token generation
def generate_next(context_words, temperature=1.0):
    """Single next-token generation step with probability bar chart."""
    ids = tf.constant([encode(' '.join(context_words))], dtype=tf.int32)
    logits_inf = model(ids, training=False)
    last = logits_inf[0, -1, :].numpy()
    probs = np.exp(last / max(temperature, 1e-6))
    probs /= probs.sum()

    topk_idx = probs.argsort()[::-1][:8]
    top_words = [IDX2WORD[int(i)] for i in topk_idx]
    top_probs = probs[topk_idx]

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.barh(top_words[::-1], top_probs[::-1],
            color=['gold' if w == top_words[0] else 'steelblue' for w in top_words[::-1]])
    ax.set_xlabel('Probability')
    ax.set_title(f'Next token probabilities | context: {context_words}  T={temperature}')
    for bar, prob in zip(ax.patches, top_probs[::-1]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9)
    plt.tight_layout(); plt.show()
    best = IDX2WORD[int(probs.argmax())]
    print(f'  Greedy prediction: "{best}"'); return best

print('=== Autoregressive generation ===\n')
context = ['the']
for step in range(5):
    print(f'Step {step+1}: context = {context}')
    next_tok = generate_next(context, temperature=0.8)
    context.append(next_tok)
    print()
print(f'Generated sequence: {" ".join(context)}')

---

## Part 10 — Why W_V Is a Relevance Filter, Not a Passthrough

$W_V$ is a **task-specific extraction lens**. Each attention layer has a different job. $W_V$ lets each layer extract exactly the slice it needs from each token's information.

In [ ]:
#  W_V as a relevance filter
embs_sentence = embedding_matrix.numpy()[TOKEN_IDS]   # (6, 3)

# W_V_action: extract ACTION channel (Animacy + Dynamism)
W_V_action = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], dtype=np.float32)
# W_V_object: extract OBJECT channel (Concreteness + Animacy)
W_V_object = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]], dtype=np.float32)

V_action = embs_sentence @ W_V_action   # (6, 2)
V_object = embs_sentence @ W_V_object   # (6, 2)

attn_row_cat = attn_w.numpy()[TOKENS.index('cat')]
blend_action = (attn_row_cat[:, None] * V_action).sum(0)
blend_object = (attn_row_cat[:, None] * V_object).sum(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
specs = [
    (axes[0], V_action, blend_action, 'W_V_action — ACTION lens\n(Animacy x Dynamism)', 'Animacy', 'Dynamism'),
    (axes[1], V_object, blend_object, 'W_V_object — OBJECT lens\n(Concreteness x Animacy)', 'Concreteness', 'Animacy'),
]
for ax, V, blend, title, xl, yl in specs:
    ax.scatter(V[:, 0], V[:, 1], s=80, c='lightgray', edgecolor='#888', zorder=3)
    for i, tok in enumerate(TOKENS):
        ax.annotate(f'[{i}]{tok}', (V[i, 0], V[i, 1] + 0.03), fontsize=8, ha='center')
    ax.scatter(*blend, s=280, color='gold', edgecolor='#b8860b', zorder=5,
               label='"cat" blended context', linewidths=2)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(title, fontsize=10); ax.legend(fontsize=9)
plt.suptitle('W_V shapes WHAT part of each token enters the weighted blend.',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('  Attention weights (W_Q/W_K path) answer WHO gets blended.')
print('  W_V answers WHAT each token contributes to that blend.')

---

## Part 11 — Causal Triangle and Accumulation Tower

The causal mask determines *how much of the sentence each position gets to know about*. Position 0 sees only itself. Position 5 sees all six tokens.

> **By the time the last position exits the final block, it has absorbed a chain of increasingly enriched representations from every earlier position.**

In [ ]:
#  Causal Triangle visualisation
mask_vis = np.tril(np.ones((SEQ_LEN, SEQ_LEN), dtype=int))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
sns.heatmap(mask_vis.astype(float), ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=TOKENS, yticklabels=TOKENS, linewidths=1.0, cbar=False,
            annot=mask_vis, fmt='d', annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Causal Mask\n1 = allowed to attend  |  0 = blocked')
ax.set_xlabel('Key token'); ax.set_ylabel('Query token')

ax2 = axes[1]
history_counts = np.arange(1, SEQ_LEN + 1)
bar_colors = plt.cm.Blues(np.linspace(0.35, 0.9, SEQ_LEN))
ax2.bar(range(SEQ_LEN), history_counts, color=bar_colors, edgecolor='white', lw=1)
ax2.set_xticks(range(SEQ_LEN)); ax2.set_xticklabels(TOKENS)
ax2.set_ylabel('Tokens visible to this position')
ax2.set_title('How many tokens each position knows about')
for i, c in enumerate(history_counts):
    ax2.text(i, c + 0.05, str(c), ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Causal Triangle: position 0 is isolated; last position absorbs all',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---

## Part 12 — Encoder Architecture: Bidirectional Attention

The only difference between an encoder block and a decoder block is **one argument**: pass `mask=None` (encoder) vs a causal mask (decoder).

**Predict:** In the encoder heatmap, how many non-zero cells will row 0 have?

A. 1 (only itself)  B. 2 (neighbours)  C. 4 (all positions)

Write your answer, then run the cell below.

In [ ]:
#  Encoder vs Decoder attention heatmap: same weights, only the mask differs
tf.random.set_seed(7)
shared_mha = layers.MultiHeadAttention(num_heads=NUM_HEADS, key_dim=D_HEAD)

x_demo = tf.random.normal([1, SEQ_LEN, D_WORK])

# Encoder: bidirectional (no mask)
enc_out, w_encoder = shared_mha(x_demo, x_demo, return_attention_scores=True)

# Decoder: causal mask
i = tf.range(SEQ_LEN)[:, tf.newaxis]
j = tf.range(SEQ_LEN)[tf.newaxis, :]
causal_bool = tf.cast(i >= j, tf.bool)   # True = attend (lower tri)
dec_out, w_decoder = shared_mha(x_demo, x_demo, attention_mask=causal_bool,
                                 return_attention_scores=True)

labels = [str(t) for t in TOKEN_IDS]
w_enc_h0 = w_encoder[0, 0].numpy()
w_dec_h0 = w_decoder[0, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, data, title, cmap in [
    (axes[0], w_enc_h0, 'Encoder (mask=None)\nBidirectional', 'Blues'),
    (axes[1], w_dec_h0, 'Decoder (causal mask)\nLower-triangle only', 'Oranges'),
]:
    sns.heatmap(data, ax=ax, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=TOKENS, yticklabels=TOKENS, linewidths=0.5,
                cbar=True, vmin=0, vmax=1, cbar_kws={'label': 'attention weight'})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Same MHA weights, same input — only the mask differs', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

nz_enc = int((w_enc_h0[0] > 0.01).sum())
nz_dec = int((w_dec_h0[0] > 0.01).sum())
print(f'Row 0 non-zero cells — Encoder: {nz_enc}   Decoder: {nz_dec}')
print(f'  -> Encoder row 0 attends to ALL {SEQ_LEN} positions  (answer: C)')
print(f'  -> Decoder row 0 attends to only 1 position (cannot see the future)')
print('\nImplementation difference: one argument — mask=None vs causal_mask.')

---

## Part 13 — Toy to Real: DistilGPT-2 with HuggingFace TF

Everything you've built used tiny dimensions so the vectors stayed readable. A production model is the **identical machinery** scaled up.

We load **DistilGPT-2** (82M parameters) using `TFAutoModel.from_pretrained` and crack it open to see its layers, weights, and generate text.

In [ ]:
#  Toy vs Real — parameter comparison
rows = [
    ('embedding dim  d_model', D_WORK, 768),
    ('attention heads',        NUM_HEADS, 12),
    ('dim per head  d_head',   D_HEAD, 64),
    ('feed-forward hidden',    D_FF, 3072),
    ('transformer layers',     2, 6),
    ('vocabulary size',        VOCAB_SIZE, 50257),
]
print(f'{"component":<24}  {"toy model":>12}  {"DistilGPT-2":>14}')
print('  ' + '-' * 54)
for name, toy, real in rows:
    print(f'  {name:<22}  {str(toy):>12}  {str(real):>14}')
print()
print('Every component in DistilGPT-2 is identical in kind to what you built.')
print('  -> Scale, not novelty, is what makes GPT-2 impressive.')

In [ ]:
#  Load DistilGPT-2 with HuggingFace (PyTorch backend — transformers 5.x dropped TF models)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print('Loading DistilGPT-2 (downloads ~350MB the first time)...')
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
gpt2 = AutoModelForCausalLM.from_pretrained('distilgpt2')
gpt2.eval()
print('Loaded!')
print()

total_params = sum(p.numel() for p in gpt2.parameters())
print(f'DistilGPT-2 total parameters: {total_params:,}')
print()
print('Top-level modules:')
for name, module in gpt2.named_children():
    print(f'  {name}')


In [ ]:
#  Inspect DistilGPT-2 internals — mirror our toy components
print('=== DistilGPT-2 Internals ===')
print()

# Token embedding shape
wte = gpt2.transformer.wte
print(f'Token embedding (wte)  : {tuple(wte.weight.shape)}')
print(f'  -> (vocab={wte.weight.shape[0]}, d_model={wte.weight.shape[1]})')
print()

# Position embedding
wpe = gpt2.transformer.wpe
print(f'Position embedding (wpe): {tuple(wpe.weight.shape)}')
print(f'  -> (max_seq={wpe.weight.shape[0]}, d_model={wpe.weight.shape[1]})')
print()

# First transformer block
block0 = gpt2.transformer.h[0]
print('Block 0 sub-modules:')
for name, module in block0.named_children():
    param_shapes = [tuple(p.shape) for p in module.parameters()]
    if param_shapes:
        print(f'  {name:<20} {param_shapes}')


In [ ]:
#  DistilGPT-2 text generation — PyTorch backend
prompt = "The cat sat on"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

with torch.no_grad():
    # Greedy generation
    output = gpt2.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=False,   # greedy
        pad_token_id=tokenizer.eos_token_id
    )
generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(f'Prompt       : {prompt!r}')
print(f'DistilGPT-2  : {generated!r}')
print()

# Temperature sampling
torch.manual_seed(42)
with torch.no_grad():
    output_sampled = gpt2.generate(
        input_ids,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
generated_s = tokenizer.decode(output_sampled[0], skip_special_tokens=True)
print(f'Sampled (T=0.8): {generated_s!r}')


---

## Architecture Comparison

| | **Encoder-only** | **Decoder-only** | **Encoder-Decoder** |
|---|---|---|---|
| **Mask** | None (bidirectional) | Causal (lower-triangle) | Enc: none; Dec: causal |
| **Attention flow** | every token ↔ every token | left → right only | cross-attention bridges |
| **Use case** | BERT, RoBERTa (classification, NLU) | GPT, LLaMA (generation) | T5, BART (seq2seq) |
| **Example** | Sentence classification | Language modelling | Translation, summarisation |

The difference is always **one** change: which positions are masked.

---

## What This Notebook Covered

| Topic | Built from scratch | Used Keras built-in |
|---|---|---|
| Vocabulary + 3D embeddings | `tf.constant`, `layers.Embedding` | `layers.Embedding` |
| Sinusoidal PE | `tf.math.sin/cos` arithmetic | — |
| RoPE | Pure numpy rotation | — |
| Scaled dot-product attention | `tf.matmul` + `tf.nn.softmax` | — |
| Multi-head attention | `tf.reshape + tf.transpose` | `layers.MultiHeadAttention` |
| Feed-forward network | `layers.Dense(gelu)` stack | — |
| Layer normalisation | `layers.LayerNormalization` | `layers.LayerNormalization` |
| Transformer block | `tf.keras.layers.Layer` subclass | — |
| Causal masking | `tf.linalg.band_part` | `use_causal_mask=True` |
| Mini LM training | `tf.GradientTape` + `tf.clip_by_global_norm` | `tf.keras.optimizers.Adam` |
| DistilGPT-2 | — | `TFAutoModelForCausalLM` |

**Adjacent topics (named but not built here):** KV-caching, sliding window attention, ALiBi, Flash Attention, speculative decoding — see the full curriculum map in `notes/`.